# Walmart Sales Intelligence
## 01 — Data Profiling, Preparation & KPI Validation

### Project Overview

This project analyzes Walmart's weekly sales data to evaluate
department-level performance and support commercial decision-making.

The analysis focuses on two business questions:

1. Which departments contribute the most to total sales?
2. How does commercial performance vary across departments and stores?

### Data Sources

The project integrates three datasets:

- `raw_ventas.csv`: Weekly sales by store and department.
- `raw_departamento.csv`: Department reference information.
- `raw_tiendas.csv`: Store characteristics and size.

### Methodology

1. Data understanding and structural inspection.
2. Data quality assessment.
3. Investigation of anomalies.
4. Data preparation and integration.
5. KPI calculation and validation.
6. Export of validated data for Tableau Studio.

**Primary analysis period:** January 6 – October 26, 2012.

**Tools:** Python, Pandas, NumPy, Jupyter Notebook,
Tableau Studio and Tableau Agent.

## 1. Environment Setup

In [1]:
# 01. Importación de librerías

import pandas as pd
import numpy as np

# Configuración de visualización
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Entorno de análisis configurado correctamente.")
print("Versión de Pandas:", pd.__version__)

Entorno de análisis configurado correctamente.
Versión de Pandas: 3.0.3


## 2. Data Loading

In [2]:
# 02. Carga de datasets originales

from pathlib import Path

# Localizar la carpeta principal del proyecto
project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

data_path = project_path / "data" / "raw"

# Cargar los archivos CSV
df_ventas = pd.read_csv(data_path / "raw_ventas.csv")
df_departamentos = pd.read_csv(data_path / "raw_departamento.csv")
df_tiendas = pd.read_csv(data_path / "raw_tiendas.csv")

print("Datasets cargados correctamente.\n")

print("Ventas:", df_ventas.shape)
print("Departamentos:", df_departamentos.shape)
print("Tiendas:", df_tiendas.shape)

Datasets cargados correctamente.

Ventas: (95880, 6)
Departamentos: (15, 2)
Tiendas: (45, 3)


## 3. Data Understanding

In [3]:
# 03. Inspección inicial de los datasets

datasets = {
    "Ventas": df_ventas,
    "Departamentos": df_departamentos,
    "Tiendas": df_tiendas
}

for nombre, df in datasets.items():

    print(f"\n{'=' * 60}")
    print(f"DATASET: {nombre.upper()}")
    print(f"{'=' * 60}")

    print("\nPrimeros 5 registros:")
    display(df.head())

    print("\nInformación estructural:")
    df.info()

    print("\nNombres de columnas:")
    print(df.columns.tolist())


DATASET: VENTAS

Primeros 5 registros:


,tienda,dept,fecha,ventas_semanales,esferiado,semana_limpia
0,1,1,2010-02-05,24924.50,False,2010-06
1,1,1,2010-02-12,46039.49,True,2010-07
2,1,1,2010-02-19,41595.55,False,2010-08
3,1,1,2010-02-26,19403.54,False,2010-09
4,1,1,2010-03-05,21827.90,False,2010-10



Información estructural:
<class 'pandas.DataFrame'>
RangeIndex: 95880 entries, 0 to 95879
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   tienda            95880 non-null  int64  
 1   dept              95880 non-null  int64  
 2   fecha             95880 non-null  str    
 3   ventas_semanales  95880 non-null  float64
 4   esferiado         95880 non-null  bool   
 5   semana_limpia     95880 non-null  str    
dtypes: bool(1), float64(1), int64(2), str(2)
memory usage: 3.7 MB

Nombres de columnas:
['tienda', 'dept', 'fecha', 'ventas_semanales', 'esferiado', 'semana_limpia']

DATASET: DEPARTAMENTOS

Primeros 5 registros:


,dept,nombre_dept
0,1,Snacks y Bebidas
1,2,Despensa y Básicos
2,3,Juguetes y Juegos
3,4,Salud y Bienestar
4,5,Electrónica de Consumo



Información estructural:
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   dept         15 non-null     int64
 1   nombre_dept  15 non-null     str  
dtypes: int64(1), str(1)
memory usage: 372.0 bytes

Nombres de columnas:
['dept', 'nombre_dept']

DATASET: TIENDAS

Primeros 5 registros:


,tienda,tipo,tamaño
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875



Información estructural:
<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   tienda  45 non-null     int64
 1   tipo    45 non-null     str  
 2   tamaño  45 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.2 KB

Nombres de columnas:
['tienda', 'tipo', 'tamaño']


## 4. Data Quality Assessment

In [4]:
# 04. Diagnóstico general de calidad de datos

for nombre, df in datasets.items():

    print(f"\n{'=' * 60}")
    print(f"DATA QUALITY: {nombre.upper()}")
    print(f"{'=' * 60}")

    # Dimensiones
    print(f"\nFilas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]}")

    # Valores nulos
    print("\nValores nulos por columna:")
    print(df.isna().sum())

    # Registros duplicados
    print("\nRegistros completamente duplicados:")
    print(df.duplicated().sum())

    # Cardinalidad
    print("\nValores únicos por columna:")
    print(df.nunique())

# Validación de unicidad en las tablas de referencia

print("\n" + "=" * 60)
print("VALIDACIÓN DE CLAVES")
print("=" * 60)

print(
    "\nDepartamentos con ID duplicado:",
    df_departamentos.duplicated(subset=["dept"]).sum()
)

print(
    "Tiendas con ID duplicado:",
    df_tiendas.duplicated(subset=["tienda"]).sum()
)

# Exploración temporal sin modificar los datos originales

fechas_diagnostico = pd.to_datetime(
    df_ventas["fecha"],
    errors="coerce",
    format="%Y-%m-%d"
)

print("\n" + "=" * 60)
print("DIAGNÓSTICO TEMPORAL")
print("=" * 60)

print("\nFecha mínima:", fechas_diagnostico.min())
print("Fecha máxima:", fechas_diagnostico.max())

print(
    "Fechas que no pudieron convertirse:",
    fechas_diagnostico.isna().sum()
)

print("\nRegistros por año:")
print(fechas_diagnostico.dt.year.value_counts().sort_index())


DATA QUALITY: VENTAS

Filas: 95,880
Columnas: 6

Valores nulos por columna:
tienda              0
dept                0
fecha               0
ventas_semanales    0
esferiado           0
semana_limpia       0
dtype: int64

Registros completamente duplicados:
0

Valores únicos por columna:
tienda                 45
dept                   15
fecha                 143
ventas_semanales    94204
esferiado               2
semana_limpia         143
dtype: int64

DATA QUALITY: DEPARTAMENTOS

Filas: 15
Columnas: 2

Valores nulos por columna:
dept           0
nombre_dept    0
dtype: int64

Registros completamente duplicados:
0

Valores únicos por columna:
dept           15
nombre_dept    15
dtype: int64

DATA QUALITY: TIENDAS

Filas: 45
Columnas: 3

Valores nulos por columna:
tienda    0
tipo      0
tamaño    0
dtype: int64

Registros completamente duplicados:
0

Valores únicos por columna:
tienda    45
tipo       3
tamaño    40
dtype: int64

VALIDACIÓN DE CLAVES

Departamentos con ID duplicado:

In [5]:
# 05. Validación de integridad referencial y granularidad

print("=" * 60)
print("VALIDACIÓN DE INTEGRIDAD REFERENCIAL")
print("=" * 60)

# Tiendas registradas en ventas que no existen en el catálogo
tiendas_sin_referencia = (
    set(df_ventas["tienda"])
    - set(df_tiendas["tienda"])
)

# Departamentos registrados en ventas que no existen en el catálogo
departamentos_sin_referencia = (
    set(df_ventas["dept"])
    - set(df_departamentos["dept"])
)

print("\nTiendas sin referencia:", len(tiendas_sin_referencia))
print("IDs:", sorted(tiendas_sin_referencia))

print("\nDepartamentos sin referencia:", len(departamentos_sin_referencia))
print("IDs:", sorted(departamentos_sin_referencia))


print("\n" + "=" * 60)
print("VALIDACIÓN DE GRANULARIDAD")
print("=" * 60)

# Comprobar si existe más de un registro para
# la misma tienda, departamento y fecha

duplicados_clave = df_ventas.duplicated(
    subset=["tienda", "dept", "fecha"]
).sum()

print(
    "\nDuplicados por tienda + departamento + fecha:",
    duplicados_clave
)


print("\n" + "=" * 60)
print("VALIDACIÓN DE TABLAS DE REFERENCIA")
print("=" * 60)

print(
    "\nIDs de departamentos duplicados:",
    df_departamentos["dept"].duplicated().sum()
)

print(
    "IDs de tiendas duplicados:",
    df_tiendas["tienda"].duplicated().sum()
)

VALIDACIÓN DE INTEGRIDAD REFERENCIAL

Tiendas sin referencia: 0
IDs: []

Departamentos sin referencia: 0
IDs: []

VALIDACIÓN DE GRANULARIDAD

Duplicados por tienda + departamento + fecha: 0

VALIDACIÓN DE TABLAS DE REFERENCIA

IDs de departamentos duplicados: 0
IDs de tiendas duplicados: 0


In [6]:
# 06. Validación de reglas de negocio

print("=" * 60)
print("VALIDACIÓN DE VENTAS")
print("=" * 60)

print(
    "\nVentas negativas:",
    (df_ventas["ventas_semanales"] < 0).sum()
)

print(
    "Ventas iguales a cero:",
    (df_ventas["ventas_semanales"] == 0).sum()
)

print(
    "Ventas positivas:",
    (df_ventas["ventas_semanales"] > 0).sum()
)

print("\nEstadísticas descriptivas:")
display(df_ventas["ventas_semanales"].describe())


print("\n" + "=" * 60)
print("VALIDACIÓN DE TIENDAS")
print("=" * 60)

print(
    "\nTiendas con tamaño cero o negativo:",
    (df_tiendas["tamaño"] <= 0).sum()
)

print("\nTipos de tiendas:")
print(df_tiendas["tipo"].value_counts(dropna=False))

print("\nEstadísticas de tamaño:")
display(df_tiendas["tamaño"].describe())


print("\n" + "=" * 60)
print("VALIDACIÓN DE SEMANAS")
print("=" * 60)

# Convertimos las fechas en una variable auxiliar
fechas = pd.to_datetime(
    df_ventas["fecha"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Calculamos la semana ISO a partir de la fecha
semanas_iso = fechas.dt.strftime("%G-%V")

# Comparamos contra la columna original
semanas_diferentes = (
    df_ventas["semana_limpia"] != semanas_iso
)

print(
    "\nRegistros con semana diferente a ISO:",
    semanas_diferentes.sum()
)

print("\nEjemplos de diferencias:")

display(
    df_ventas.loc[
        semanas_diferentes,
        ["fecha", "semana_limpia"]
    ].assign(
        semana_iso=semanas_iso[semanas_diferentes]
    ).head(10)
)

VALIDACIÓN DE VENTAS

Ventas negativas: 27
Ventas iguales a cero: 0
Ventas positivas: 95853

Estadísticas descriptivas:


count     95880.000000
mean      19943.079625
std       19032.569715
min       -4988.940000
25%        6628.002500
50%       15177.705000
75%       27448.982500
max      406988.630000
Name: ventas_semanales, dtype: float64


VALIDACIÓN DE TIENDAS

Tiendas con tamaño cero o negativo: 0

Tipos de tiendas:
tipo
A    22
B    17
C     6
Name: count, dtype: int64

Estadísticas de tamaño:


count        45.000000
mean     130287.600000
std       63825.271991
min       34875.000000
25%       70713.000000
50%      126512.000000
75%      202307.000000
max      219622.000000
Name: tamaño, dtype: float64


VALIDACIÓN DE SEMANAS

Registros con semana diferente a ISO: 95880

Ejemplos de diferencias:


,fecha,semana_limpia,semana_iso
0,2010-02-05,2010-06,2010-05
1,2010-02-12,2010-07,2010-06
2,2010-02-19,2010-08,2010-07
3,2010-02-26,2010-09,2010-08
4,2010-03-05,2010-10,2010-09
5,2010-03-12,2010-11,2010-10
6,2010-03-19,2010-12,2010-11
7,2010-03-26,2010-13,2010-12
8,2010-04-02,2010-14,2010-13
9,2010-04-09,2010-15,2010-14


### 4.1 Anomaly Investigation

In [7]:
# 07. Investigación de anomalías de calidad

print("=" * 60)
print("ANÁLISIS DE VENTAS NEGATIVAS")
print("=" * 60)

# Identificar registros negativos
ventas_negativas = df_ventas[
    df_ventas["ventas_semanales"] < 0
].copy()

# Añadir el año para analizar su distribución
ventas_negativas["año"] = pd.to_datetime(
    ventas_negativas["fecha"]
).dt.year

print("\nVentas negativas por año:")
print(
    ventas_negativas["año"]
    .value_counts()
    .sort_index()
)

print("\nImpacto económico total:")
print(ventas_negativas["ventas_semanales"].sum())

print("\nRegistros negativos:")
display(
    ventas_negativas.sort_values(
        "ventas_semanales"
    ).head(10)
)


print("\n" + "=" * 60)
print("ANÁLISIS DE CONSISTENCIA SEMANAL")
print("=" * 60)

# Crear una tabla temporal de comparación
comparacion_semanas = pd.DataFrame({
    "fecha": fechas,
    "semana_original": df_ventas["semana_limpia"],
    "semana_iso": fechas.dt.strftime("%G-%V")
})

# Obtener combinaciones únicas para evitar
# repetir miles de veces la misma semana
comparacion_semanas = (
    comparacion_semanas
    .drop_duplicates()
    .sort_values("fecha")
)

print("\nPrimeras 10 semanas:")
display(comparacion_semanas.head(10))

print("\nÚltimas 10 semanas:")
display(comparacion_semanas.tail(10))

print("\nNúmero de semanas distintas:")
print(len(comparacion_semanas))

# Comprobar si semana_limpia utiliza
# el número de semana de %U (domingo como inicio)
semanas_u = fechas.dt.strftime("%Y-%U")

# Comprobar %W (lunes como inicio)
semanas_w = fechas.dt.strftime("%Y-%W")

print("\nCoincidencias con convención %U:")
print((df_ventas["semana_limpia"] == semanas_u).sum())

print("\nCoincidencias con convención %W:")
print((df_ventas["semana_limpia"] == semanas_w).sum())

print("\nCoincidencias con calendario ISO:")
print(
    (
        df_ventas["semana_limpia"]
        == fechas.dt.strftime("%G-%V")
    ).sum()
)

ANÁLISIS DE VENTAS NEGATIVAS

Ventas negativas por año:
año
2010     3
2011    14
2012    10
Name: count, dtype: int64

Impacto económico total:
-7077.39

Registros negativos:


,tienda,dept,fecha,ventas_semanales,esferiado,semana_limpia,año
58665,28,6,2010-10-08,-4988.94,False,2010-41,2010
74890,35,16,2012-08-24,-771.90,False,2012-35,2012
74894,35,16,2012-09-21,-244.17,False,2012-39,2012
58660,28,6,2010-09-03,-157.63,False,2010-36,2010
79719,38,6,2010-12-10,-153.12,False,2010-50,2010
846,1,6,2012-08-10,-139.65,False,2012-33,2012
92831,44,9,2011-12-16,-132.46,False,2011-51,2011
45861,22,6,2012-01-13,-106.34,False,2012-03,2012
9388,5,6,2011-11-18,-101.26,False,2011-47,2011
52276,25,6,2011-08-26,-58.56,False,2011-35,2011



ANÁLISIS DE CONSISTENCIA SEMANAL

Primeras 10 semanas:


,fecha,semana_original,semana_iso
0,2010-02-05,2010-06,2010-05
1,2010-02-12,2010-07,2010-06
2,2010-02-19,2010-08,2010-07
3,2010-02-26,2010-09,2010-08
4,2010-03-05,2010-10,2010-09
5,2010-03-12,2010-11,2010-10
6,2010-03-19,2010-12,2010-11
7,2010-03-26,2010-13,2010-12
8,2010-04-02,2010-14,2010-13
9,2010-04-09,2010-15,2010-14



Últimas 10 semanas:


,fecha,semana_original,semana_iso
133,2012-08-24,2012-35,2012-34
134,2012-08-31,2012-36,2012-35
135,2012-09-07,2012-37,2012-36
136,2012-09-14,2012-38,2012-37
137,2012-09-21,2012-39,2012-38
138,2012-09-28,2012-40,2012-39
139,2012-10-05,2012-41,2012-40
140,2012-10-12,2012-42,2012-41
141,2012-10-19,2012-43,2012-42
142,2012-10-26,2012-44,2012-43



Número de semanas distintas:
143

Coincidencias con convención %U:
0

Coincidencias con convención %W:
0

Coincidencias con calendario ISO:
0


## 5. Data Preparation

In [8]:
# 08. Investigación del desplazamiento semanal

print("=" * 60)
print("ANÁLISIS DEL DESPLAZAMIENTO SEMANAL")
print("=" * 60)

# Tabla temporal con fechas y semanas originales
df_semanas = df_ventas[
    ["fecha", "semana_limpia"]
].copy()

# Convertir fecha
df_semanas["fecha"] = pd.to_datetime(
    df_semanas["fecha"],
    format="%Y-%m-%d"
)

# Extraer componentes ISO
calendario_iso = df_semanas["fecha"].dt.isocalendar()

df_semanas["año_iso"] = calendario_iso.year.astype(int)
df_semanas["numero_semana_iso"] = calendario_iso.week.astype(int)

# Extraer componentes de la semana original
componentes = df_semanas["semana_limpia"].str.extract(
    r"^(\d{4})-(\d{2})$"
)

df_semanas["año_original"] = pd.to_numeric(
    componentes[0],
    errors="coerce"
)

df_semanas["numero_semana_original"] = pd.to_numeric(
    componentes[1],
    errors="coerce"
)

# Calcular desplazamiento
df_semanas["diferencia_semanas"] = (
    df_semanas["numero_semana_original"]
    - df_semanas["numero_semana_iso"]
)

print("\nDistribución de diferencias:")
print(
    df_semanas["diferencia_semanas"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRegistros donde la diferencia NO es +1:")

excepciones = df_semanas[
    df_semanas["diferencia_semanas"] != 1
]

print("Cantidad:", len(excepciones))

display(
    excepciones.drop_duplicates(
        subset=["fecha", "semana_limpia"]
    ).head(20)
)

print("\nRegistros con año original distinto del año ISO:")

años_diferentes = df_semanas[
    df_semanas["año_original"] != df_semanas["año_iso"]
]

print("Cantidad:", len(años_diferentes))

display(
    años_diferentes.drop_duplicates(
        subset=["fecha", "semana_limpia"]
    ).head(20)
)

ANÁLISIS DEL DESPLAZAMIENTO SEMANAL

Distribución de diferencias:
diferencia_semanas
1    95880
Name: count, dtype: int64

Registros donde la diferencia NO es +1:
Cantidad: 0


,fecha,semana_limpia,año_iso,numero_semana_iso,año_original,numero_semana_original,diferencia_semanas



Registros con año original distinto del año ISO:
Cantidad: 0


,fecha,semana_limpia,año_iso,numero_semana_iso,año_original,numero_semana_original,diferencia_semanas


In [9]:
# 09. Preparación inicial de datos

print("=" * 60)
print("PREPARACIÓN DE DATOS")
print("=" * 60)

# Crear una copia para preservar los datos originales
df_ventas_clean = df_ventas.copy()

# Convertir la columna fecha a datetime
df_ventas_clean["fecha"] = pd.to_datetime(
    df_ventas_clean["fecha"],
    format="%Y-%m-%d",
    errors="raise"
)

# Generar calendario ISO
calendario_iso = df_ventas_clean["fecha"].dt.isocalendar()

# Crear columnas temporales estandarizadas
df_ventas_clean["año_iso"] = calendario_iso.year.astype(int)

df_ventas_clean["semana_iso"] = (
    calendario_iso.year.astype(str)
    + "-"
    + calendario_iso.week.astype(str).str.zfill(2)
)

# Crear columna de año calendario
df_ventas_clean["año"] = (
    df_ventas_clean["fecha"].dt.year
)

# Validar las transformaciones
print("\nTipos de datos:")
print(df_ventas_clean.dtypes)

print("\nPrimeros registros:")
display(df_ventas_clean.head())

print("\nCantidad original de registros:", len(df_ventas))

print(
    "Cantidad de registros preparados:",
    len(df_ventas_clean)
)

print(
    "\nValores nulos en fecha:",
    df_ventas_clean["fecha"].isna().sum()
)

PREPARACIÓN DE DATOS

Tipos de datos:
tienda                       int64
dept                         int64
fecha               datetime64[us]
ventas_semanales           float64
esferiado                     bool
semana_limpia                  str
año_iso                      int64
semana_iso                     str
año                          int32
dtype: object

Primeros registros:


,tienda,dept,fecha,ventas_semanales,esferiado,semana_limpia,año_iso,semana_iso,año
0,1,1,2010-02-05,24924.50,False,2010-06,2010,2010-05,2010
1,1,1,2010-02-12,46039.49,True,2010-07,2010,2010-06,2010
2,1,1,2010-02-19,41595.55,False,2010-08,2010,2010-07,2010
3,1,1,2010-02-26,19403.54,False,2010-09,2010,2010-08,2010
4,1,1,2010-03-05,21827.90,False,2010-10,2010,2010-09,2010



Cantidad original de registros: 95880
Cantidad de registros preparados: 95880

Valores nulos en fecha: 0


## 6. Data Integration

In [10]:
# 10. Integración controlada de datasets

print("=" * 60)
print("INTEGRACIÓN DE DATOS")
print("=" * 60)

# Registrar condiciones iniciales
filas_originales = len(df_ventas_clean)
ventas_originales = df_ventas_clean["ventas_semanales"].sum()

# Integrar información de departamentos
df_integrado = df_ventas_clean.merge(
    df_departamentos,
    on="dept",
    how="left",
    validate="many_to_one"
)

# Integrar información de tiendas
df_integrado = df_integrado.merge(
    df_tiendas,
    on="tienda",
    how="left",
    validate="many_to_one"
)

# Validaciones posteriores
filas_finales = len(df_integrado)
ventas_finales = df_integrado["ventas_semanales"].sum()

print("\nRegistros originales:", filas_originales)
print("Registros integrados:", filas_finales)

print("\nVentas originales:", round(ventas_originales, 2))
print("Ventas integradas:", round(ventas_finales, 2))

print("\nDiferencia de registros:", filas_finales - filas_originales)

print(
    "Diferencia de ventas:",
    round(ventas_finales - ventas_originales, 2)
)

print("\nValores nulos en columnas integradas:")

print(
    df_integrado[
        ["nombre_dept", "tipo", "tamaño"]
    ].isna().sum()
)

print("\nPrimeros registros de la tabla integrada:")

display(df_integrado.head())

INTEGRACIÓN DE DATOS

Registros originales: 95880
Registros integrados: 95880

Ventas originales: 1912142474.48
Ventas integradas: 1912142474.48

Diferencia de registros: 0
Diferencia de ventas: 0.0

Valores nulos en columnas integradas:
nombre_dept    0
tipo           0
tamaño         0
dtype: int64

Primeros registros de la tabla integrada:


,tienda,dept,fecha,ventas_semanales,esferiado,semana_limpia,año_iso,semana_iso,año,nombre_dept,tipo,tamaño
0,1,1,2010-02-05,24924.50,False,2010-06,2010,2010-05,2010,Snacks y Bebidas,A,151315
1,1,1,2010-02-12,46039.49,True,2010-07,2010,2010-06,2010,Snacks y Bebidas,A,151315
2,1,1,2010-02-19,41595.55,False,2010-08,2010,2010-07,2010,Snacks y Bebidas,A,151315
3,1,1,2010-02-26,19403.54,False,2010-09,2010,2010-08,2010,Snacks y Bebidas,A,151315
4,1,1,2010-03-05,21827.90,False,2010-10,2010,2010-09,2010,Snacks y Bebidas,A,151315


## 7. Business KPI Analysis

In [11]:
# 11. Definición del período y KPIs generales

print("=" * 60)
print("ANÁLISIS COMERCIAL - WALMART 2012")
print("=" * 60)

# Crear una copia exclusiva de 2012
df_2012 = df_integrado[
    df_integrado["año"] == 2012
].copy()

# Validar período disponible
fecha_min = df_2012["fecha"].min()
fecha_max = df_2012["fecha"].max()

print("\nPERÍODO ANALIZADO")
print("Fecha inicial:", fecha_min.date())
print("Fecha final:", fecha_max.date())

print("\nRegistros:", len(df_2012))

print(
    "Semanas distintas:",
    df_2012["fecha"].nunique()
)

# Calcular indicadores generales
ventas_totales = df_2012["ventas_semanales"].sum()

tiendas_activas = df_2012["tienda"].nunique()

departamentos_activos = df_2012["dept"].nunique()

ventas_promedio_registro = (
    df_2012["ventas_semanales"].mean()
)

# Mostrar resultados
print("\nKPIs GENERALES")

print(f"Ventas totales: ${ventas_totales:,.2f}")

print("Tiendas activas:", tiendas_activas)

print("Departamentos activos:", departamentos_activos)

print(
    f"Venta promedio por registro: "
    f"${ventas_promedio_registro:,.2f}"
)

# Validar registros negativos en 2012
ventas_negativas_2012 = (
    df_2012["ventas_semanales"] < 0
).sum()

print(
    "\nRegistros negativos en 2012:",
    ventas_negativas_2012
)

ANÁLISIS COMERCIAL - WALMART 2012

PERÍODO ANALIZADO
Fecha inicial: 2012-01-06
Fecha final: 2012-10-26

Registros: 28845
Semanas distintas: 43

KPIs GENERALES
Ventas totales: $558,411,520.58
Tiendas activas: 45
Departamentos activos: 15
Venta promedio por registro: $19,359.04

Registros negativos en 2012: 10


### 7.1 Department Sales Contribution

In [12]:
# 12. KPI: Participación de ventas por departamento

print("=" * 60)
print("PARTICIPACIÓN DE VENTAS POR DEPARTAMENTO - 2012")
print("=" * 60)

# Agrupar ventas por departamento
df_participacion = (
    df_2012
    .groupby(
        ["dept", "nombre_dept"],
        as_index=False
    )
    .agg(
        ventas_totales=("ventas_semanales", "sum"),
        registros=("ventas_semanales", "size"),
        tiendas_activas=("tienda", "nunique")
    )
)

# Calcular participación porcentual
df_participacion["participacion_pct"] = (
    df_participacion["ventas_totales"]
    / ventas_totales
    * 100
)

# Ordenar de mayor a menor participación
df_participacion = (
    df_participacion
    .sort_values(
        "participacion_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

# Mostrar resultados
display(
    df_participacion.round(2)
)

# Validaciones
print("\nVALIDACIONES")

print(
    "Total de departamentos:",
    len(df_participacion)
)

print(
    "Suma de participaciones (%):",
    round(
        df_participacion["participacion_pct"].sum(),
        4
    )
)

print(
    "Diferencia frente a ventas originales:",
    round(
        df_participacion["ventas_totales"].sum()
        - ventas_totales,
        2
    )
)

PARTICIPACIÓN DE VENTAS POR DEPARTAMENTO - 2012


,dept,nombre_dept,ventas_totales,registros,tiendas_activas,participacion_pct
0,2,Despensa y Básicos,85052985.88,1935,45,15.23
1,13,Comida Fresca,59510812.89,1935,45,10.66
2,8,Artículos del Hogar y Papel,58876083.76,1935,45,10.54
3,4,Salud y Bienestar,51003352.42,1935,45,9.13
4,9,Ropa,40183258.61,1925,45,7.20
5,7,Hogar y Temporada,37849580.53,1935,45,6.78
6,10,Cuidado del Bebé y la Familia,35790072.73,1935,45,6.41
7,1,Snacks y Bebidas,35445927.76,1935,45,6.35
8,5,Electrónica de Consumo,33472398.70,1935,45,5.99
9,16,Departamento No Identificado,29431276.26,1935,45,5.27



VALIDACIONES
Total de departamentos: 15
Suma de participaciones (%): 100.0
Diferencia frente a ventas originales: 0.0


### 7.2 Department Coverage

In [13]:
# 13. Validación de cobertura por departamento - 2012

print("=" * 60)
print("COBERTURA DE DEPARTAMENTOS - 2012")
print("=" * 60)

# Número de semanas observadas en el período
semanas_periodo = df_2012["fecha"].nunique()

# Número total de tiendas del catálogo
total_tiendas = df_tiendas["tienda"].nunique()

# Calcular cobertura por departamento
df_cobertura = (
    df_2012
    .groupby(["dept", "nombre_dept"], as_index=False)
    .agg(
        registros_observados=("fecha", "size"),
        tiendas_activas=("tienda", "nunique"),
        semanas_observadas=("fecha", "nunique")
    )
)

# Registros potenciales bajo cobertura completa
df_cobertura["registros_esperados"] = (
    total_tiendas * semanas_periodo
)

# Calcular porcentaje de cobertura
df_cobertura["cobertura_pct"] = (
    df_cobertura["registros_observados"]
    / df_cobertura["registros_esperados"]
    * 100
)

# Calcular combinaciones ausentes
df_cobertura["registros_faltantes"] = (
    df_cobertura["registros_esperados"]
    - df_cobertura["registros_observados"]
)

# Ordenar por menor cobertura
df_cobertura = (
    df_cobertura
    .sort_values("cobertura_pct")
    .reset_index(drop=True)
)

display(df_cobertura.round(2))

print("\nVALIDACIONES")

print("Tiendas del catálogo:", total_tiendas)
print("Semanas del período:", semanas_periodo)

print(
    "Combinaciones potenciales por departamento:",
    total_tiendas * semanas_periodo
)

print(
    "Total de combinaciones ausentes:",
    df_cobertura["registros_faltantes"].sum()
)

COBERTURA DE DEPARTAMENTOS - 2012


,dept,nombre_dept,registros_observados,tiendas_activas,semanas_observadas,registros_esperados,cobertura_pct,registros_faltantes
0,6,Jardín y Vida al Aire Libre,1765,43,43,1935,91.21,170
1,9,Ropa,1925,45,43,1935,99.48,10
2,3,Juguetes y Juegos,1935,45,43,1935,100.00,0
3,1,Snacks y Bebidas,1935,45,43,1935,100.00,0
4,4,Salud y Bienestar,1935,45,43,1935,100.00,0
5,5,Electrónica de Consumo,1935,45,43,1935,100.00,0
6,7,Hogar y Temporada,1935,45,43,1935,100.00,0
7,2,Despensa y Básicos,1935,45,43,1935,100.00,0
8,8,Artículos del Hogar y Papel,1935,45,43,1935,100.00,0
9,10,Cuidado del Bebé y la Familia,1935,45,43,1935,100.00,0



VALIDACIONES
Tiendas del catálogo: 45
Semanas del período: 43
Combinaciones potenciales por departamento: 1935
Total de combinaciones ausentes: 180


### 7.3 Coverage Investigation

In [14]:
# 14. Investigación de cobertura por tienda y departamento

print("=" * 60)
print("COBERTURA POR TIENDA Y DEPARTAMENTO")
print("=" * 60)

# Número de semanas disponibles en 2012
total_semanas = df_2012["fecha"].nunique()

# Filtrar los departamentos con cobertura incompleta
departamentos_revisar = [6, 9]

# Contar las semanas observadas por tienda y departamento
cobertura_tienda = (
    df_2012[
        df_2012["dept"].isin(departamentos_revisar)
    ]
    .groupby(
        ["dept", "nombre_dept", "tienda"],
        as_index=False
    )
    .agg(
        semanas_con_registro=("fecha", "nunique"),
        ventas_totales=("ventas_semanales", "sum")
    )
)

# Crear todas las combinaciones posibles
# entre las 45 tiendas y los dos departamentos
combinaciones = pd.MultiIndex.from_product(
    [
        departamentos_revisar,
        df_tiendas["tienda"].unique()
    ],
    names=["dept", "tienda"]
).to_frame(index=False)

# Integrar las combinaciones con los datos observados
cobertura_completa = combinaciones.merge(
    cobertura_tienda,
    on=["dept", "tienda"],
    how="left",
    validate="one_to_one"
)

# Las combinaciones sin registros tienen 0 semanas observadas
cobertura_completa["semanas_con_registro"] = (
    cobertura_completa["semanas_con_registro"]
    .fillna(0)
    .astype(int)
)

# Calcular semanas sin registro
cobertura_completa["semanas_sin_registro"] = (
    total_semanas
    - cobertura_completa["semanas_con_registro"]
)

# Mostrar únicamente tiendas con cobertura incompleta
resultado = cobertura_completa[
    cobertura_completa["semanas_sin_registro"] > 0
].sort_values(
    ["dept", "semanas_sin_registro"],
    ascending=[True, False]
)

print("\nTiendas con cobertura incompleta:")
display(resultado)

print("\nVALIDACIÓN")

print(
    "Total de combinaciones ausentes:",
    resultado["semanas_sin_registro"].sum()
)

COBERTURA POR TIENDA Y DEPARTAMENTO

Tiendas con cobertura incompleta:


,dept,tienda,nombre_dept,semanas_con_registro,ventas_totales,semanas_sin_registro
32,6,33,NaN,0,NaN,43
41,6,42,NaN,0,NaN,43
35,6,36,Jardín y Vida al Aire Libre,2,22.76,41
42,6,43,Jardín y Vida al Aire Libre,23,452.64,20
29,6,30,Jardín y Vida al Aire Libre,35,596.48,8
43,6,44,Jardín y Vida al Aire Libre,36,1011.80,7
36,6,37,Jardín y Vida al Aire Libre,37,1114.81,6
37,6,38,Jardín y Vida al Aire Libre,41,1294.52,2
80,9,36,Ropa,36,5222.00,7
77,9,33,Ropa,40,1842.45,3



VALIDACIÓN
Total de combinaciones ausentes: 180


### 7.4 Commercial Performance Indicators

In [15]:
# 15. Indicadores de eficiencia comercial - 2012

print("=" * 60)
print("EFICIENCIA COMERCIAL POR DEPARTAMENTO")
print("=" * 60)

# Agrupar información por departamento
df_eficiencia = (
    df_2012
    .groupby(["dept", "nombre_dept"], as_index=False)
    .agg(
        ventas_totales=("ventas_semanales", "sum"),
        registros=("ventas_semanales", "size"),
        tiendas_activas=("tienda", "nunique"),
        tamaño_promedio=("tamaño", "mean")
    )
)

# KPI solicitado por TripleTen
df_eficiencia["ventas_por_tamaño_promedio"] = (
    df_eficiencia["ventas_totales"]
    / df_eficiencia["tamaño_promedio"]
)

# Indicador complementario:
# ventas promedio por tienda-semana observada
df_eficiencia["ventas_por_tienda_semana"] = (
    df_eficiencia["ventas_totales"]
    / df_eficiencia["registros"]
)

# Ordenar por el indicador solicitado
df_eficiencia = (
    df_eficiencia
    .sort_values(
        "ventas_por_tamaño_promedio",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nINDICADORES POR DEPARTAMENTO")

display(
    df_eficiencia.round(2)
)

print("\nVALIDACIONES")

print(
    "Departamentos analizados:",
    len(df_eficiencia)
)

print(
    "Diferencia de ventas:",
    round(
        df_eficiencia["ventas_totales"].sum()
        - ventas_totales,
        2
    )
)

print(
    "Tamaños promedio iguales o menores a cero:",
    (df_eficiencia["tamaño_promedio"] <= 0).sum()
)

EFICIENCIA COMERCIAL POR DEPARTAMENTO

INDICADORES POR DEPARTAMENTO


,dept,nombre_dept,ventas_totales,registros,tiendas_activas,tamaño_promedio,ventas_por_tamaño_promedio,ventas_por_tienda_semana
0,2,Despensa y Básicos,85052985.88,1935,45,130287.60,652.81,43955.03
1,13,Comida Fresca,59510812.89,1935,45,130287.60,456.76,30754.94
2,8,Artículos del Hogar y Papel,58876083.76,1935,45,130287.60,451.89,30426.92
3,4,Salud y Bienestar,51003352.42,1935,45,130287.60,391.47,26358.32
4,9,Ropa,40183258.61,1925,45,130757.44,307.31,20874.42
5,7,Hogar y Temporada,37849580.53,1935,45,130287.60,290.51,19560.51
6,10,Cuidado del Bebé y la Familia,35790072.73,1935,45,130287.60,274.70,18496.16
7,1,Snacks y Bebidas,35445927.76,1935,45,130287.60,272.06,18318.31
8,5,Electrónica de Consumo,33472398.70,1935,45,130287.60,256.91,17298.40
9,16,Departamento No Identificado,29431276.26,1935,45,130287.60,225.89,15209.96



VALIDACIONES
Departamentos analizados: 15
Diferencia de ventas: 0.0
Tamaños promedio iguales o menores a cero: 0


## 8. KPI Validation

In [16]:
# 16. Validación cruzada de KPIs comerciales

import numpy as np

print("=" * 60)
print("VALIDACIÓN FINAL DE KPIs - WALMART 2012")
print("=" * 60)

# 1. Validación de ventas totales

ventas_detalle = df_2012["ventas_semanales"].sum()

ventas_departamentos = (
    df_participacion["ventas_totales"].sum()
)

ventas_eficiencia = (
    df_eficiencia["ventas_totales"].sum()
)

assert np.isclose(
    ventas_detalle,
    ventas_departamentos,
    atol=0.01,
    rtol=0
), "Error: ventas no coinciden con participación"

assert np.isclose(
    ventas_detalle,
    ventas_eficiencia,
    atol=0.01,
    rtol=0
), "Error: ventas no coinciden con eficiencia"

print("\nVentas totales validadas:")
print(f"{ventas_detalle:,.2f}")

# 2. Validación de participación

participacion_total = (
    df_participacion["participacion_pct"].sum()
)

assert np.isclose(
    participacion_total,
    100,
    atol=0.0001
), "Error en participación porcentual"

print("\nParticipación total:")
print(f"{participacion_total:.4f}%")

# 3. Validación de cantidad de registros

registros_departamentos = (
    df_participacion["registros"].sum()
)

assert registros_departamentos == len(df_2012), (
    "Error: diferencia en cantidad de registros"
)

print("\nRegistros validados:")
print(registros_departamentos)

# 4. Validación de indicadores de eficiencia

assert (
    df_eficiencia["tamaño_promedio"] > 0
).all(), "Error: tamaño promedio inválido"

assert (
    df_eficiencia["registros"] > 0
).all(), "Error: departamentos sin registros"

# 5. Validación de cobertura

assert (
    df_cobertura["registros_observados"].sum()
    == len(df_2012)
), "Error en cobertura de departamentos"

print("\nCobertura validada correctamente.")

print("\n" + "=" * 60)
print("RESULTADO: VALIDACIONES SUPERADAS")
print("=" * 60)

VALIDACIÓN FINAL DE KPIs - WALMART 2012

Ventas totales validadas:
558,411,520.58

Participación total:
100.0000%

Registros validados:
28845

Cobertura validada correctamente.

RESULTADO: VALIDACIONES SUPERADAS


## 9. Results and Conclusions

### 9.1 Dataset Overview

The analysis integrates three datasets containing weekly sales,
department information, and store characteristics.

| Metric | Result |
|---|---|
| Total sales records | 95,880 |
| Stores | 45 |
| Departments | 15 |
| Historical period | 2010–2012 |
| Analysis period | January 6 – October 26, 2012 |
| Records analyzed (2012) | 28,845 |
| Weeks analyzed (2012) | 43 |

### 9.2 Data Quality Findings

The following validations were performed:

| Quality dimension | Finding | Action |
|---|---|---|
| Completeness | No null values detected | No imputation required |
| Duplicate records | No duplicates identified | No records removed |
| Referential integrity | All store and department IDs matched | Relationships validated |
| Primary key uniqueness | No duplicates in reference tables | Many-to-one joins validated |
| Granularity | Unique store-department-date combinations | Transaction grain preserved |
| Negative sales | 27 records identified (10 in 2012) | Retained; business cause unconfirmed |
| Store size | No zero or negative values | No corrections required |
| Weekly consistency | Original week labels differ from ISO weeks by +1 | Standardized ISO week created |
| Department coverage | 180 potential store-department-week combinations absent in 2012 | No artificial records generated |

### 9.3 Data Cleaning Decisions

The following transformations were implemented:

1. Converted the `fecha` column to datetime.
2. Created standardized ISO calendar fields.
3. Preserved the original `semana_limpia` column for traceability.
4. Integrated sales, department, and store datasets using validated many-to-one joins.
5. Retained negative sales values because their business meaning could not be confirmed.
6. Preserved historical records while creating a separate analytical subset for 2012.

No sales records were removed during preparation.

### 9.4 Business KPI Validation

The following results were independently calculated using Python:

| KPI | Validated result |
|---|---:|
| Total sales (2012 YTD) | 558,411,520.58 |
| Active stores | 45 |
| Active departments | 15 |
| Department sales participation | 100% |
| Highest sales department | Despensa y Básicos |
| Highest department participation | 15.23% |
| Negative sales records (2012) | 10 |

All implemented reconciliation checks passed.

### 9.5 Key Business Findings

**Finding 1 — Sales concentration**

Despensa y Básicos generated 15.23% of total sales.

The five highest-revenue departments collectively accounted for
approximately 52.76% of sales.

**Finding 2 — Department coverage**

Jardín y Vida al Aire Libre had the lowest observed coverage
(91.21%) and the lowest average sales per observed store-week.

The available data does not establish whether missing combinations
represent inactivity, unavailable departments, or missing records.

**Finding 3 — KPI interpretation**

The academic sales-to-average-store-size ratio was reproduced,
but it should not be interpreted as actual department-level
sales per square meter.

Department-specific floor-space allocation is unavailable,
and the measurement unit of store size has not been confirmed.

### 9.6 Limitations

- The 2012 dataset covers only January through October.
- Store size does not represent department-specific floor space.
- Negative sales may represent returns or accounting adjustments.
- The business meaning of missing department observations is unknown.
- One department is labeled as unidentified.
- Revenue analysis does not establish profitability because cost information is unavailable.
- The currency of the sales values has not been independently verified.

### 9.7 Conclusion

The integrated dataset passed the implemented structural,
referential integrity, and KPI reconciliation checks.

The data is suitable for exploratory sales analysis and dashboard
development, subject to the documented business and coverage limitations.

The next phase will use Tableau Studio and Tableau Agent to generate
an executive dashboard and validate AI-generated metrics against
the Python reference results.

## 10. Data Export

In [18]:
# 17. Exportación de datos procesados para Tableau Studio

from pathlib import Path

print("=" * 60)
print("EXPORTACIÓN DE DATOS - WALMART")
print("=" * 60)

# Localizar la carpeta principal del proyecto
project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

# Crear carpeta de salida
output_path = project_path / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)

# Definir archivos
archivo_historico = output_path / "walmart_sales_clean.csv"
archivo_2012 = output_path / "walmart_sales_2012.csv"
archivo_kpis = output_path / "walmart_department_kpis_2012.csv"

# Exportar datos históricos
df_integrado.to_csv(
    archivo_historico,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

# Exportar datos de 2012
df_2012.to_csv(
    archivo_2012,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

# Integrar los KPIs validados
df_kpis = df_participacion.merge(
    df_eficiencia[
        [
            "dept",
            "tamaño_promedio",
            "ventas_por_tamaño_promedio",
            "ventas_por_tienda_semana"
        ]
    ],
    on="dept",
    how="left",
    validate="one_to_one"
)

# Exportar KPIs
df_kpis.to_csv(
    archivo_kpis,
    index=False,
    encoding="utf-8-sig"
)

# Verificar archivos generados
print("\nARCHIVOS GENERADOS:")

for archivo in [
    archivo_historico,
    archivo_2012,
    archivo_kpis
]:
    print(f"\n{archivo.name}")
    print(f"Tamaño: {archivo.stat().st_size / 1024:.2f} KB")

# Validar exportaciones
historico_exportado = pd.read_csv(archivo_historico)
ventas_2012_exportadas = pd.read_csv(archivo_2012)
kpis_exportados = pd.read_csv(archivo_kpis)

assert len(historico_exportado) == 95880
assert len(ventas_2012_exportadas) == 28845
assert len(kpis_exportados) == 15

assert np.isclose(
    ventas_2012_exportadas["ventas_semanales"].sum(),
    558411520.58,
    atol=0.01,
    rtol=0
)

print("\nRESULTADO: EXPORTACIÓN VALIDADA CORRECTAMENTE")

EXPORTACIÓN DE DATOS - WALMART

ARCHIVOS GENERADOS:

walmart_sales_clean.csv
Tamaño: 8180.25 KB

walmart_sales_2012.csv
Tamaño: 2461.23 KB

walmart_department_kpis_2012.csv
Tamaño: 1.75 KB

RESULTADO: EXPORTACIÓN VALIDADA CORRECTAMENTE
